In [ ]:
import requests

In [ ]:
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.feishu import base_fetch_func,fetch_tenant_token

In [ ]:
bitapp_auth={'app_id':'cli_a1fc0a192db8100c','app_secret':'nNk1fT8h2eVi8B2ymIFangHTpYrOAJTD'}
usertoken='u-0Z3mOz6b5cJVSQwaL21eHb1kmyT5hluzV0G0lgqw2J29'


In [ ]:
fetch_tenant_token(auth=bitapp_auth)


In [ ]:
def build_headers():
    
    token=fetch_tenant_token(bitapp_auth)
    #token='u-3atGsodiN9391_BlvmfeZh1kkwt5hlQxo0G0hgew2Ijc'
    headers = {
        'Authorization':
        'Bearer ' + token,  # your access token
        #'Content-Type': 'application/json; charset=utf-8'
    }
    return headers
build_headers()

In [ ]:
def fetch_app_tables(app_token):
    headers=build_headers()
    url='https://open.feishu.cn/open-apis/bitable/v1/apps/%s/tables'%app_token
    r=base_fetch_func(url,headers,method='get')
    return r
tables=fetch_app_tables('bascnx0n63ZppXLCFaSVKT0arSg')
tables

In [ ]:
bitable_works(app_token)

In [ ]:
def fetch_table_records(app_token,table_id,**kwargs):
    '''reference:https://open.feishu.cn/document/uAjLw4CM/ukTMukTMukTM/reference/bitable-v1/app-table-record/list
    路径参数，kwargs:
    filter 示例值："AND(CurrentValue.[身高]>180, CurrentValue.[体重]>150)"
    field_names 字段名称，用于指定本次查询返回记录中包含的字段 示例值："["字段1","字段2"]"
    page_token 分页标记，第一次请求不填，表示从头开始遍历；分页查询结果还有更多项时会同时返回新的 page_token，下次遍历可采用该 page_token 获取查询结果 示例值："recn0hoyXL"         
    page_size  默认值：20 数据校验规则：最大值：500
    
    '''
    
    headers=build_headers()
    url='https://open.feishu.cn/open-apis/bitable/v1/apps/%s/tables/%s/records'%(app_token,table_id)
    
    r=base_fetch_func(url,headers=headers,params=kwargs,method='get')
    #print(r)
    return r

app_token='bascnx0n63ZppXLCFaSVKT0arSg'
table_id='tblPlTiBMOFD7suF'


In [ ]:
from urllib.parse import quote,unquote

In [ ]:
unquote('%5B%22%E8%8A%82%E7%82%B9%E5%90%8D%E7%A7%B0%22%2C%22%E8%8A%82%E7%82%B9%E7%B1%BB%E5%9E%8B%22%2C%22%E5%85%AC%E5%8F%B8%E4%BB%A3%E7%A0%81%22%2C%22%E7%94%BB%E5%B8%83%E4%BD%8D%E7%BD%AEX%22%2C%22%E7%94%BB%E5%B8%83%E4%BD%8D%E7%BD%AEY%22%2C%22%E5%8F%82%E4%B8%8E%E5%AD%90%E5%9B%BE%22%2C%22%E6%89%80%E5%B1%9E%E5%A4%96%E9%83%A8%E9%AA%A8%E5%B9%B2%22%5D')

In [ ]:
# filter=quote('CurrentValue.[参与子图].contains("粗骨干图")'),field_names=quote(["节点名称","节点类型"])
r=fetch_table_records(app_token,table_id,filter='CurrentValue.[参与子图].contains("粗骨干图")',field_names='["节点名称","节点类型"]')
print(r)

In [ ]:
type(r['data']['has_more'])

In [ ]:
import requests

In [ ]:

r.json()
print(len(r.json()['data']))

In [ ]:
r.json()

In [ ]:
data=r.json()['data']
data

In [ ]:
def bitable_check(app_token):
    try:
        res=requests.get(r'http://172.21.0.14:81/bitapp/tables?app_token=%s'%app_token)
        tables=res.json()
        if tables['code']==0:
            #print(tables)
            tables=tables['data']
            node_table,link_table,indi_table=None,None,None

            for table in tables:
                if table['name']=='节点表':
                    node_table=table['table_id']
                if table['name']=='关系表':
                    link_table=table['table_id']
                if table['name']=='监测指标表':
                    indi_table=table['table_id']
            if node_table and link_table and indi_table:
                print('good table')
                return True,[node_table,link_table,indi_table]
            else:
                msg='not all tables found, node,link,indicator :%s'%[node_table,link_table,indi_table]
                return False,msg
        else:
            print(tables)
            return False,tables
    except Exception as e:
        return False,e

In [ ]:
bitable_check('bascnx0n63ZppXLCFaSVKT0arSg')

In [ ]:
from flask import jsonify

In [ ]:
def bitable_works(bitapp_token,subgraph_name=None):
    '''查多维表格的完整性（节点表 关系表 监测指标表'''
    res,data=bitable_check(bitapp_token)
    if not res:
        return jsonify(code=-1,msg='查多维表格完整性失败，'+data)
    else:
        #查节点表内容
        node_table,link_table,indi_table=data
        nodes,links,indicators=None,None,None
        params={'app_token':bitapp_token,'table_id':node_table,'graph_name':subgraph_name,'fields':'["节点名称","节点类型","公司代码","画布位置X","画布位置Y","参与子图","所属外部骨干"]'}
        r=base_fetch_func('http://172.21.0.14:81/bitapp/record',headers=None,params=params,method='get')
        if r['code']==0:
            nodes=r['data']
        params={'app_token':bitapp_token,'table_id':link_table,'graph_name':subgraph_name,'fields':'["主体","关系","客体","参与子图"]'}
        r=base_fetch_func('http://172.21.0.14:81/bitapp/record',headers=None,params=params,method='get')
        if r['code']==0:
            links=r['data']       
        params={'app_token':bitapp_token,'table_id':indi_table,'graph_name':subgraph_name,'fields':'["指标名","指标代码","指标性质","单位","指标定义","影响方向","画布位置X","画布位置Y","世界线跟踪表-主要相关指标"]'}
        r=base_fetch_func('http://172.21.0.14:81/bitapp/record',headers=None,params=params,method='get')
        if r['code']==0:
            indicators=r['data']   
        if nodes and links and indicators:
            result={'nodes':nodes,'links':links,'indicators':indicators}
            return result
        else:
            return False
          

In [ ]:
r=bitable_works('bascnx0n63ZppXLCFaSVKT0arSg','粗骨干图')

In [ ]:
r

In [114]:
nodes=[{
      "fields": {
        "参与子图": [
          {
            "record_ids": [
              "rec7txIH74"
            ], 
            "table_id": "tblXDUFR2NRoGLTC", 
            "text": "粗骨干图", 
            "text_arr": [
              "粗骨干图"
            ], 
            "type": "text"
          }
        ], 
        "所属外部骨干": [
          {
            "record_ids": [
              "recGA9FEeQ"
            ], 
            "table_id": "tblFqSdp0hqRNyLy", 
            "text": "储能", 
            "text_arr": [
              "储能"
            ], 
            "type": "text"
          }
        ], 
        "节点名称": "某外部储能产品", 
        "节点类型": "产品"
      }, 
      "id": "rectJnewZG", 
      "record_id": "rectJnewZG"
    }, 
    {
      "fields": {
        "参与子图": [
          {
            "record_ids": [
              "recFANk4QN"
            ], 
            "table_id": "tblXDUFR2NRoGLTC", 
            "text": "某局部图1", 
            "text_arr": [
              "某局部图1"
            ], 
            "type": "text"
          }
        ], 
        "画布位置X": "35.21", 
        "画布位置Y": "15.21", 
        "节点名称": "风电场", 
        "节点类型": "产品"
      }, 
      "id": "rec86jSDlY", 
      "record_id": "rec86jSDlY"
    }, ]
nodes.append({
        "fields": {
        "公司代码": "600905.SH", 
        "参与子图": [
          {
            "record_ids": [
              "recFANk4QN", 
              "rec7txIH74"
            ], 
            "table_id": "tblXDUFR2NRoGLTC", 
            "text": "某局部图1,粗骨干图", 
            "text_arr": [
              "某局部图1", 
              "粗骨干图"
            ], 
            "type": "text"
          }
        ], 
        "节点名称": "三峡能源", 
        "节点类型": "企业"
      }, 
      "id": "rec9KgK3RV", 
      "record_id": "rec9KgK3RV"
    },)
nodes

[{'fields': {'参与子图': [{'record_ids': ['rec7txIH74'],
     'table_id': 'tblXDUFR2NRoGLTC',
     'text': '粗骨干图',
     'text_arr': ['粗骨干图'],
     'type': 'text'}],
   '所属外部骨干': [{'record_ids': ['recGA9FEeQ'],
     'table_id': 'tblFqSdp0hqRNyLy',
     'text': '储能',
     'text_arr': ['储能'],
     'type': 'text'}],
   '节点名称': '某外部储能产品',
   '节点类型': '产品'},
  'id': 'rectJnewZG',
  'record_id': 'rectJnewZG'},
 {'fields': {'参与子图': [{'record_ids': ['recFANk4QN'],
     'table_id': 'tblXDUFR2NRoGLTC',
     'text': '某局部图1',
     'text_arr': ['某局部图1'],
     'type': 'text'}],
   '画布位置X': '35.21',
   '画布位置Y': '15.21',
   '节点名称': '风电场',
   '节点类型': '产品'},
  'id': 'rec86jSDlY',
  'record_id': 'rec86jSDlY'},
 {'fields': {'公司代码': '600905.SH',
   '参与子图': [{'record_ids': ['recFANk4QN', 'rec7txIH74'],
     'table_id': 'tblXDUFR2NRoGLTC',
     'text': '某局部图1,粗骨干图',
     'text_arr': ['某局部图1', '粗骨干图'],
     'type': 'text'}],
   '节点名称': '三峡能源',
   '节点类型': '企业'},
  'id': 'rec9KgK3RV',
  'record_id': 'rec9KgK3RV'}]

In [115]:

links=[{
      "fields": {
        "主体": [
          {
            "record_ids": [
              "rec86jSDlY"
            ], 
            "table_id": "tblPlTiBMOFD7suF", 
            "text": "风电场", 
            "text_arr": [
              "风电场"
            ], 
            "type": "text"
          }
        ], 
        "关系": [
          {
            "record_ids": [
              "rec1cMkjOT"
            ], 
            "table_id": "tblXBqHmlnY21CWH", 
            "text": "父节点", 
            "text_arr": [
              "父节点"
            ], 
            "type": "text"
          }
        ], 
        "客体": [
          {
            "record_ids": [
              "recoH0xpBJ"
            ], 
            "table_id": "tblPlTiBMOFD7suF", 
            "text": "海上风电场", 
            "text_arr": [
              "海上风电场"
            ], 
            "type": "text"
          }
        ]
      }, 
      "id": "reczeDqu7Q", 
      "record_id": "reczeDqu7Q"
    }, 
    {
      "fields": {
        "主体": [
          {
            "record_ids": [
              "recE3vQsGy"
            ], 
            "table_id": "tblPlTiBMOFD7suF", 
            "text": "东方电缆", 
            "text_arr": [
              "东方电缆"
            ], 
            "type": "text"
          }
        ], 
        "关系": [
          {
            "record_ids": [
              "rect7AB6SO"
            ], 
            "table_id": "tblXBqHmlnY21CWH", 
            "text": "生产", 
            "text_arr": [
              "生产"
            ], 
            "type": "text"
          }
        ], 
        "客体": [
          {
            "record_ids": [
              "rectJnewZG"
            ], 
            "table_id": "tblPlTiBMOFD7suF", 
            "text": "某外部储能产品", 
            "text_arr": [
              "某外部储能产品"
            ], 
            "type": "text"
          }
        ]
      }, 
      "id": "recyESjWnP", 
      "record_id": "recyESjWnP"
    }]
len(links)
links[0]

{'fields': {'主体': [{'record_ids': ['rec86jSDlY'],
    'table_id': 'tblPlTiBMOFD7suF',
    'text': '风电场',
    'text_arr': ['风电场'],
    'type': 'text'}],
  '关系': [{'record_ids': ['rec1cMkjOT'],
    'table_id': 'tblXBqHmlnY21CWH',
    'text': '父节点',
    'text_arr': ['父节点'],
    'type': 'text'}],
  '客体': [{'record_ids': ['recoH0xpBJ'],
    'table_id': 'tblPlTiBMOFD7suF',
    'text': '海上风电场',
    'text_arr': ['海上风电场'],
    'type': 'text'}]},
 'id': 'reczeDqu7Q',
 'record_id': 'reczeDqu7Q'}

In [116]:
def get_main_info(field):
    '''从某双向选择的field中获取textarr项，去除其他'''
    '''双选字段信息为list但似乎只有1个元素，处理方式存疑'''
    #return [one['text_arr'] for one in field]
    return field[0]['text_arr'] if field else None
get_main_info(nodes[4]['fields']['参与子图'])

IndexError: list index out of range

In [117]:
res_links=[]
for l in links:
    f=l['fields']
    res_link={'id':l['record_id'],'source':f['主体'][0]['record_ids'][0],'target':f['客体'][0]['record_ids'][0],'category':get_main_info(f['关系'])[0]}
    res_links.append(res_link)
res_links

[{'id': 'reczeDqu7Q',
  'source': 'rec86jSDlY',
  'target': 'recoH0xpBJ',
  'category': '父节点'},
 {'id': 'recyESjWnP',
  'source': 'recE3vQsGy',
  'target': 'rectJnewZG',
  'category': '生产'}]

In [118]:
res_nodes=[]
for n in nodes:
    f=n['fields']
    more_info={}
    code=f.get('公司代码')
    if code:more_info['code']=code
    external_backbone=get_main_info(f.get('所属外部骨干'))
    if external_backbone:more_info['external_backbone']=external_backbone

    res_node={'id':n['record_id'],'name':f.get('节点名称'),'category':f.get('节点类型'),'more_info':more_info}
    x,y=f.get('画布位置X'),f.get('画布位置Y')
    if x and y:
        res_node['coordinates']=[x,y]
    res_nodes.append(res_node)
res_nodes
    

[{'id': 'rectJnewZG',
  'name': '某外部储能产品',
  'category': '产品',
  'more_info': {'external_backbone': ['储能']}},
 {'id': 'rec86jSDlY',
  'name': '风电场',
  'category': '产品',
  'more_info': {},
  'coordinates': ['35.21', '15.21']},
 {'id': 'rec9KgK3RV',
  'name': '三峡能源',
  'category': '企业',
  'more_info': {'code': '600905.SH'}}]

1


In [ ]:
res_nodes

In [101]:
indicators=[{
      "fields": {
        "挂钩节点": [
          {
            "record_ids": [
              "rec86jSDlY"
            ], 
            "table_id": "tblPlTiBMOFD7suF", 
            "text": "风电场", 
            "text_arr": [
              "风电场"
            ], 
            "type": "text"
          }
        ], 
        "指标名": "全国新增风电装机量", 
        "指标性质": "定量"
      }, 
      "id": "recgJNitOI", 
      "record_id": "recgJNitOI"
    }]

In [122]:

for i in indicators:
    f=i['fields']
    res_link={'id':i['record_id'],'source':i['record_id'],'target':f['挂钩节点'][0]['record_ids'][0],'category':"反映"}
    #res_links.append(res_link)
    #"指标代码","挂钩节点","指标性质","单位","指标定义","影响方向","画布位置X","画布位置Y","世界线跟踪表-主要相关指标"
    more_info={}
    type=f.get('指标性质')
    if type:more_info['type']=type
    code=f.get('指标代码')
    if code:more_info['code']=code
    res_node={'id':i['record_id'],'name':f.get('指标名'),'category':'指标','more_info':more_info}
    
    x,y=f.get('画布位置X'),f.get('画布位置Y')
    if x and y:
        res_node['coordinates']=[x,y]
    res_links.append(res_link)
    res_nodes.append(res_node)
print(res_node)
print(res_link)

{'id': 'recgJNitOI', 'name': '全国新增风电装机量', 'category': '指标', 'more_info': {'type': '定量'}}
{'id': 'recgJNitOI', 'source': 'recgJNitOI', 'target': 'rec86jSDlY', 'category': '反映'}
